<a href="https://colab.research.google.com/github/vccf/Deep-Learning-Experiments/blob/drafts-TPH-YOLOv5/Demo3-TPH-YOLOv5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pycocotools

In [ ]:
!pip install -q grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 66.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

#import os
#WORK_DIR = '/content/drive/MyDrive/tph_yolov5_visdrone'
#os.makedirs(WORK_DIR, exist_ok=True)
#os.chdir(WORK_DIR)
#print("Working directory:", os.getcwd())

Mounted at /content/drive
Working directory: /content/drive/MyDrive/tph_yolov5_visdrone


In [ ]:
import os
WORK_DIR = '/content/work'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print("Working directory:", os.getcwd())

Working directory: /content/work


In [ ]:
!git clone https://github.com/cv516Buaa/tph-yolov5.git
%cd tph-yolov5

# TPH-YOLOv5 was built against an older YOLOv5/torch stack; pin versions
# that are known to work to avoid silent API mismatches (e.g. autoshape,
# torch.meshgrid 'indexing' kwarg, numpy>=1.24 removing np.float, etc.)
!pip install -q "numpy<1.24" "Pillow>=9.0" "PyYAML>=5.3.1" "scipy>=1.4.1" \
                 "tqdm>=4.41.0" "matplotlib>=3.2.2" "seaborn>=0.11.0" \
                 "opencv-python>=4.1.2" "pandas>=1.1.4" "requests>=2.23.0" \
                 "tensorboard>=2.4.1"
!pip install -q -r requirements.txt

Cloning into 'tph-yolov5'...
remote: Enumerating objects: 246, done.
remote: Counting objects: 100% (246/246), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 246 (delta 106), reused 218 (delta 97), pack-reused 0 (from 0)
Receiving objects: 100% (246/246), 9.27 MiB | 16.83 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/work/tph-yolov5
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 76.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a pro

In [ ]:
# Patch requirements.txt to skip torch/torchvision reinstall
!sed -i '/^torch/Id' requirements.txt
!sed -i '/^torchvision/Id' requirements.txt

In [ ]:
import os
os.makedirs('datasets/VisDrone', exist_ok=True)
%cd datasets/VisDrone

# Official mirrors (GitHub releases of the VisDrone dataset)
urls = {
    "VisDrone2019-DET-train.zip": "https://drive.google.com/uc?id=1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn",
    "VisDrone2019-DET-val.zip":   "https://drive.google.com/uc?id=1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59",
    "VisDrone2019-DET-test-dev.zip": "https://drive.google.com/uc?id=1PFdW_VFSCfZ_sTSZAGjQdifF_Xd5mf0V",
}

!pip install -q gdown
import gdown
for fname, url in urls.items():
    if not os.path.exists(fname):
        gdown.download(url, fname, quiet=False)
    !unzip -q -o "{fname}"
%cd ../..

/content/work/tph-yolov5/datasets/VisDrone


Downloading...
From (original): https://drive.google.com/uc?id=1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn
From (redirected): https://drive.google.com/uc?id=1a2oHjcEcwXP8oUF95qiwrqzACb2YlUhn&confirm=t&uuid=97c731d8-1a13-4766-9a60-0638cef6f870
To: /content/work/tph-yolov5/datasets/VisDrone/VisDrone2019-DET-train.zip
100%|██████████| 1.55G/1.55G [00:23<00:00, 66.0MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59
From (redirected): https://drive.google.com/uc?id=1bxK5zgLn0_L8x276eKkuYA_FzwCIjb59&confirm=t&uuid=3d01965e-c75f-4a71-aa52-7ccd6ff927d7
To: /content/work/tph-yolov5/datasets/VisDrone/VisDrone2019-DET-val.zip
100%|██████████| 81.6M/81.6M [00:01<00:00, 73.3MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1PFdW_VFSCfZ_sTSZAGjQdifF_Xd5mf0V
From (redirected): https://drive.google.com/uc?id=1PFdW_VFSCfZ_sTSZAGjQdifF_Xd5mf0V&confirm=t&uuid=0bd6f4dc-f7a0-48d9-a2b8-59af706cc918
To: /content/work/tph-yolov5/datasets/VisDrone/Vi

/content/work/tph-yolov5


In [ ]:
import os
from PIL import Image
from tqdm import tqdm

# VisDrone's 10 real object classes (categories 1-10); 0 = ignored, 11 = other
VISDRONE_CLASSES = [
    'pedestrian', 'people', 'bicycle', 'car', 'van', 'truck',
    'tricycle', 'awning-tricycle', 'bus', 'motor'
]

def convert_split(split_dir):
    img_dir = os.path.join(split_dir, 'images')
    ann_dir = os.path.join(split_dir, 'annotations')
    label_dir = os.path.join(split_dir, 'labels')
    os.makedirs(label_dir, exist_ok=True)

    for ann_file in tqdm(os.listdir(ann_dir), desc=f"Converting {split_dir}"):
        if not ann_file.endswith('.txt'):
            continue
        img_name = ann_file.replace('.txt', '.jpg')
        img_path = os.path.join(img_dir, img_name)
        if not os.path.exists(img_path):
            continue
        with Image.open(img_path) as im:
            w, h = im.size

        out_lines = []
        with open(os.path.join(ann_dir, ann_file), 'r') as f:
            for line in f:
                parts = line.strip().split(',')
                if len(parts) < 6:
                    continue
                x, y, bw, bh, score, cat = map(int, parts[:6])
                cat = int(cat)
                if cat == 0 or cat == 11 or bw <= 0 or bh <= 0:
                    continue  # skip ignored regions and 'other'
                cls_id = cat - 1  # remap 1-10 -> 0-9
                xc = (x + bw / 2) / w
                yc = (y + bh / 2) / h
                nw = bw / w
                nh = bh / h
                # clip to [0,1] in case of annotation noise near image edges
                xc, yc, nw, nh = [min(max(v, 0.0), 1.0) for v in (xc, yc, nw, nh)]
                out_lines.append(f"{cls_id} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}")

        with open(os.path.join(label_dir, ann_file), 'w') as f:
            f.write('\n'.join(out_lines))

base = 'datasets/VisDrone'
convert_split(f'{base}/VisDrone2019-DET-train')
convert_split(f'{base}/VisDrone2019-DET-val')
convert_split(f'{base}/VisDrone2019-DET-test-dev')

Converting datasets/VisDrone/VisDrone2019-DET-train: 100%|██████████| 6471/6471 [00:02<00:00, 2330.03it/s]
Converting datasets/VisDrone/VisDrone2019-DET-val: 100%|██████████| 548/548 [00:00<00:00, 2004.57it/s]


FileNotFoundError: [Errno 2] No such file or directory: 'datasets/VisDrone/VisDrone2019-DET-test-dev/annotations'

In [ ]:
import os
base = 'datasets/VisDrone'
for split in ['VisDrone2019-DET-train', 'VisDrone2019-DET-val', 'VisDrone2019-DET-test-dev']:
    p = os.path.join(base, split)
    print(p, "exists:", os.path.exists(p))
    if os.path.exists(p):
        print("  contents:", os.listdir(p))

# also check the raw zip
print(os.listdir(base))

datasets/VisDrone/VisDrone2019-DET-train exists: True
  contents: ['labels', 'annotations', 'images']
datasets/VisDrone/VisDrone2019-DET-val exists: True
  contents: ['labels', 'annotations', '.DS_Store', 'images']
datasets/VisDrone/VisDrone2019-DET-test-dev exists: True
  contents: ['labels']
['VisDrone2019-DET-val', 'VisDrone2019-DET-test-dev.zip', 'VisDrone2019-DET-train.zip', 'VisDrone2019-DET-train', 'annotations', 'VisDrone2019-DET-test-dev', 'VisDrone2019-DET-val.zip', 'images']


In [ ]:
import shutil, os

base = 'datasets/VisDrone'
target = os.path.join(base, 'VisDrone2019-DET-test-dev')

for folder in ['annotations', 'images']:
    src = os.path.join(base, folder)
    dst = os.path.join(target, folder)
    if os.path.exists(src):
        shutil.move(src, dst)

print(os.listdir(target))

['labels', 'annotations', 'images']


In [ ]:
convert_split(f'{base}/VisDrone2019-DET-test-dev')

Converting datasets/VisDrone/VisDrone2019-DET-test-dev: 100%|██████████| 1610/1610 [00:00<00:00, 2532.96it/s]


In [ ]:
dataset_yaml = """
train: datasets/VisDrone/VisDrone2019-DET-train/images
val: datasets/VisDrone/VisDrone2019-DET-val/images
test: datasets/VisDrone/VisDrone2019-DET-test-dev/images

nc: 10
names: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck',
        'tricycle', 'awning-tricycle', 'bus', 'motor']
"""

with open('data/VisDrone.yaml', 'w') as f:
    f.write(dataset_yaml)

print(open('data/VisDrone.yaml').read())


train: datasets/VisDrone/VisDrone2019-DET-train/images
val: datasets/VisDrone/VisDrone2019-DET-val/images
test: datasets/VisDrone/VisDrone2019-DET-test-dev/images

nc: 10
names: ['pedestrian', 'people', 'bicycle', 'car', 'van', 'truck',
        'tricycle', 'awning-tricycle', 'bus', 'motor']



In [ ]:
!ls models/*.yaml | grep -i tph

models/yolov5l-tph-plus.yaml
models/yolov5l-xs-tph.yaml


In [ ]:
!pwd

/content/work/tph-yolov5


In [ ]:
import re, glob

py_files = glob.glob('**/*.py', recursive=True)
patched = 0

for fpath in py_files:
    with open(fpath, 'r') as f:
        content = f.read()
    # only touch torch.load(...) calls that don't already specify weights_only
    if 'torch.load(' in content and 'weights_only' not in content:
        new_content = re.sub(
            r'torch\.load\(([^)]*)\)',
            lambda m: f'torch.load({m.group(1)}, weights_only=False)',
            content
        )
        if new_content != content:
            with open(fpath, 'w') as f:
                f.write(new_content)
            patched += 1
            print("Patched:", fpath)

print(f"\nTotal files patched: {patched}")

Patched: detect.py
Patched: hubconf.py
Patched: train.py
Patched: models/experimental.py
Patched: utils/general.py
Patched: utils/loggers/__init__.py
Patched: utils/aws/resume.py

Total files patched: 7


In [ ]:
import re, glob

py_files = glob.glob('**/*.py', recursive=True)
patched = 0

# map deprecated numpy aliases -> their modern replacement
replacements = {
    r'\bnp\.int\b': 'int',
    r'\bnp\.float\b': 'float',
    r'\bnp\.bool\b': 'bool',
    r'\bnp\.object\b': 'object',
    r'\bnp\.str\b': 'str',
}

for fpath in py_files:
    with open(fpath, 'r') as f:
        content = f.read()
    new_content = content
    for pattern, repl in replacements.items():
        new_content = re.sub(pattern, repl, new_content)
    if new_content != content:
        with open(fpath, 'w') as f:
            f.write(new_content)
        patched += 1
        print("Patched:", fpath)

print(f"\nTotal files patched: {patched}")

Patched: utils/datasets.py
Patched: utils/general.py

Total files patched: 2


In [ ]:
fpath = '/content/work/tph-yolov5/utils/loss.py'
with open(fpath, 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if 'gj.clamp_' in line or 'gi.clamp_' in line:
        print(f"Line {i+1}: {line}", end='')
        # replace in-place clamp_ with out-of-place clamp + explicit long cast
        lines[i] = line \
            .replace('gj.clamp_(0, gain[3] - 1)', '(gj.clamp(0, int(gain[3]) - 1)).long()') \
            .replace('gi.clamp_(0, gain[2] - 1)', '(gi.clamp(0, int(gain[2]) - 1)).long()') \
            .replace('.long().long()', '.long()')  # avoid double .long() if prev patch partially applied
        print(f"  → fixed to: {lines[i]}", end='')

with open(fpath, 'w') as f:
    f.writelines(lines)

print("\nDone. Verify:")
print(lines[239])  # line 240 (0-indexed = 239)

Line 240:             indices.append((b, a, gj.clamp_(0, gain[3] - 1).long(), gi.clamp_(0, gain[2] - 1).long()))  # image, anchor, grid indices
  → fixed to:             indices.append((b, a, (gj.clamp(0, int(gain[3]) - 1)).long(), (gi.clamp(0, int(gain[2]) - 1)).long()))  # image, anchor, grid indices

Done. Verify:
            indices.append((b, a, (gj.clamp(0, int(gain[3]) - 1)).long(), (gi.clamp(0, int(gain[2]) - 1)).long()))  # image, anchor, grid indices



In [ ]:
import re

fpath = '/content/work/tph-yolov5/utils/loss.py'
with open(fpath, 'r') as f:
    content = f.read()

# cast gj and gi to long AFTER clamping, instead of relying on implicit cast
old = "indices.append((b, a, gj.clamp_(0, gain[3] - 1), gi.clamp_(0, gain[2] - 1)))"
new = "indices.append((b, a, gj.clamp_(0, gain[3] - 1).long(), gi.clamp_(0, gain[2] - 1).long()))"

new_content = content.replace(old, new)
assert new_content != content, "Pattern not found — check the line manually"

with open(fpath, 'w') as f:
    f.write(new_content)

print("Patched loss.py")

Patched loss.py


In [ ]:
fpath = '/content/work/tph-yolov5/train.py'
with open(fpath, 'r') as f:
    content = f.read()

new_content = content.replace(
    "amp.autocast(enabled=cuda)",
    "torch.amp.autocast('cuda', enabled=cuda)"
)

with open(fpath, 'w') as f:
    f.write(new_content)

print("Patched train.py")

Patched train.py


In [ ]:
import re, glob

py_files = glob.glob('/content/work/tph-yolov5/**/*.py', recursive=True)

patches = {
    # numpy deprecated aliases
    r'\bnp\.int\b': 'int',
    r'\bnp\.float\b': 'float',
    r'\bnp\.bool\b': 'bool',
    r'\bnp\.object\b': 'object',
    r'\bnp\.str\b': 'str',
    r'\bnp\.complex\b': 'complex',
    # autocast
    r'amp\.autocast\(enabled=cuda\)': "torch.amp.autocast('cuda', enabled=cuda)",
}

for fpath in py_files:
    with open(fpath, 'r') as f:
        content = f.read()
    new_content = content
    for pattern, repl in patches.items():
        new_content = re.sub(pattern, repl, new_content)
    if new_content != content:
        with open(fpath, 'w') as f:
            f.write(new_content)
        print("Patched:", fpath)

print("Sweep done")

Sweep done


In [ ]:
!python train.py \
    --img 640 \
    --batch 2 \
    --epochs 10 \
    --data data/VisDrone.yaml \
    --cfg models/yolov5l-xs-tph.yaml \
    --weights yolov5l.pt \
    --hyp data/hyps/hyp.scratch.yaml \
    --name visdrone_tph_exp \
    --cache disk

2026-07-03 05:28:02.329171: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=yolov5l.pt, cfg=models/yolov5l-xs-tph.yaml, data=data/VisDrone.yaml, hyp=data/hyps/hyp.scratch.yaml, epochs=10, batch_size=2, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, evolve=None, bucket=, cache=disk, image_weights=False, device=, multi_scale=False, single_cls=False, adam=False, sync_bn=False, workers=8, project=runs/train, name=visdro

In [ ]:
import re, glob

py_files = glob.glob('/content/work/tph-yolov5/**/*.py', recursive=True)
patched = 0

for fpath in py_files:
    with open(fpath, 'r') as f:
        content = f.read()

    # Fix the bad patch: torch.device('cpu', weights_only=False) -> torch.device('cpu'), weights_only=False
    new_content = re.sub(
        r"torch\.load\(([^)]*?),\s*weights_only=False\)",
        lambda m: f"torch.load({m.group(1)}, weights_only=False)",
        content
    )

    # First fix any torch.device(..., weights_only=False) that got mangled
    new_content = re.sub(
        r"torch\.device\(([^)]*?),\s*weights_only=False\)",
        r"torch.device(\1)",
        new_content
    )

    # Now ensure every torch.load() has weights_only=False as a top-level arg
    new_content = re.sub(
        r"torch\.load\(([^)]*?)(?:,\s*weights_only=False)?\)",
        lambda m: (
            f"torch.load({m.group(1)}, weights_only=False)"
            if 'weights_only' not in m.group(1)
            else m.group(0)
        ),
        new_content
    )

    if new_content != content:
        with open(fpath, 'w') as f:
            f.write(new_content)
        patched += 1
        print("Patched:", fpath)

print(f"\nTotal files patched: {patched}")


Total files patched: 0


In [ ]:
!python train.py --resume runs/train/visdrone_tph_exp/weights/last.pt

In [ ]:
#-img 1536 \ 1536
#--batch 8 \ 4
#--epochs 100 \ 80

In [ ]:
!python val.py \
    --weights runs/train/visdrone_tph_exp/weights/best.pt \
    --data data/VisDrone.yaml \
    --img 640 \
    --task test \
    --batch 2 \
    --conf-thres 0.25 \
    --iou-thres 0.45 \
    --save-txt \
    --save-conf \
    --name visdrone_tph_test_eval

val: data=data/VisDrone.yaml, weights=['runs/train/visdrone_tph_exp/weights/best.pt'], batch_size=2, imgsz=640, conf_thres=0.25, iou_thres=0.45, task=test, device=, single_cls=False, augment=False, verbose=False, save_txt=True, save_hybrid=False, save_conf=True, save_json=False, project=runs/val, name=visdrone_tph_test_eval, exist_ok=False, half=False
YOLOv5 🚀 052dfeb torch 2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)

Traceback (most recent call last):
  File "/content/work/tph-yolov5/val.py", line 359, in <module>
    main(opt)
  File "/content/work/tph-yolov5/val.py", line 333, in main
    run(**vars(opt))
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/_contextlib.py", line 124, in decorate_context
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/content/work/tph-yolov5/val.py", line 124, in run
    model = attempt_load(weights, map_location=device)  # load FP32 model
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/work/tph-

In [ ]:
fpath = '/content/work/tph-yolov5/models/experimental.py'
with open(fpath, 'r') as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if 'attempt_download' in line and 'weights_only' in line:
        print(f"Line {i+1} before: {line}", end='')
        lines[i] = line.replace(
            'attempt_download(w, weights_only=False)',
            'attempt_download(w)'
        )
        # make sure torch.load itself still has weights_only=False
        if 'weights_only=False' not in lines[i]:
            lines[i] = lines[i].replace(
                'torch.load(attempt_download(w),',
                'torch.load(attempt_download(w),'
            ).replace(
                'map_location=map_location)',
                'map_location=map_location, weights_only=False)'
            )
        print(f"Line {i+1} after:  {lines[i]}", end='')

with open(fpath, 'w') as f:
    f.writelines(lines)

Line 96 before:         ckpt = torch.load(attempt_download(w, weights_only=False), map_location=map_location)  # load
Line 96 after:          ckpt = torch.load(attempt_download(w), map_location=map_location, weights_only=False)  # load


In [ ]:
from IPython.display import Image as IPImage, display

display(IPImage('runs/val/visdrone_tph_test_eval/confusion_matrix.png'))
display(IPImage('runs/val/visdrone_tph_test_eval/PR_curve.png'))

FileNotFoundError: No such file or directory: 'runs/val/visdrone_tph_test_eval/confusion_matrix.png'

FileNotFoundError: No such file or directory: 'runs/val/visdrone_tph_test_eval/confusion_matrix.png'

<IPython.core.display.Image object>

FileNotFoundError: No such file or directory: 'runs/val/visdrone_tph_test_eval/PR_curve.png'

FileNotFoundError: No such file or directory: 'runs/val/visdrone_tph_test_eval/PR_curve.png'

<IPython.core.display.Image object>

In [ ]:
import torch
ckpt_results = torch.load('runs/train/visdrone_tph_exp/weights/best.pt', map_location='cpu')
print(ckpt_results.keys())  # best.pt stores training metadata, not per-class P/R

# The authoritative per-class numbers are parsed from val.py's stdout table,
# or you can re-run val.py with --save-json and parse runs/val/.../best_predictions.json
# against the ground truth using pycocotools for a fully programmatic readout:

In [ ]:
import cv2
import numpy as np
import torch
from pytorch_grad_cam import EigenCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
import sys

sys.path.append('.')  # so we can import the repo's model-loading utilities
from models.experimental import attempt_load

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = attempt_load('runs/train/visdrone_tph_exp/weights/best.pt', map_location=device)
model.eval()

# Pick one test image
test_img_dir = 'datasets/VisDrone/VisDrone2019-DET-test-dev/images'
img_name = sorted(os.listdir(test_img_dir))[0]  # change index to pick a specific image
img_path = os.path.join(test_img_dir, img_name)
print("Using:", img_path)

img_size = 1536
img_bgr = cv2.imread(img_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img_rgb, (img_size, img_size))
rgb_float = np.float32(img_resized) / 255.0

input_tensor = torch.from_numpy(rgb_float).permute(2, 0, 1).unsqueeze(0).to(device)

# Target layer: the last layer of the backbone/neck before the detection heads
# is typically most semantically rich. For TPH-YOLOv5's model.model is a
# Sequential of layers; print it once to pick an index if this doesn't match.
print(model.model[-2])  # sanity check on what we're hooking
target_layers = [model.model[-2]]

class YOLOEigenCAMWrapper(torch.nn.Module):
    """EigenCAM needs a model whose forward() returns a single tensor;
    YOLO's raw forward returns a tuple (predictions, raw_features_per_head).
    We forward only the first element so grad-cam's internals don't choke."""
    def __init__(self, yolo_model):
        super().__init__()
        self.yolo_model = yolo_model

    def forward(self, x):
        out = self.yolo_model(x)
        return out[0] if isinstance(out, tuple) else out

wrapped_model = YOLOEigenCAMWrapper(model).to(device).eval()

cam = EigenCAM(model=wrapped_model, target_layers=target_layers)
grayscale_cam = cam(input_tensor)[0, :, :]

cam_image = show_cam_on_image(rgb_float, grayscale_cam, use_rgb=True)

import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(img_resized)
axes[0].set_title(f"Original: {img_name}")
axes[0].axis('off')
axes[1].imshow(cam_image)
axes[1].set_title("EigenCAM")
axes[1].axis('off')
plt.tight_layout()
plt.savefig('eigencam_result.png', dpi=150)
plt.show()